# TruthShield 02: Train + evaluate (in parts)
Needs notebook 01 finished (`cache/hifi/features.pt` + `faces.npy` on Drive). T4 GPU.

Every trained model (mode × seed) is **one part**, saved to Drive as soon as it is done
(`runs/` for weights, `results/parts/` for scores). Finished parts are skipped, so you can
stop at any time and continue later. The **merge** cells build the final tables from
whatever parts exist.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, glob
ROOT = '/content/drive/MyDrive/TruthShield'
%cd /content
if not os.path.exists('/content/TruthShield'):
    !git clone -q https://github.com/Jawad374/TruthShield.git /content/TruthShield
!git -C /content/TruthShield pull -q
%cd /content/TruthShield
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Copy the packed cache to fast local disk (faces.npy is a few GB; reading it from Drive is slow)
CACHE = '/content/cache'
os.makedirs(CACHE, exist_ok=True)
for f in ['features.pt', 'faces.npy']:
    src, dst = f'{ROOT}/cache/hifi/{f}', f'{CACHE}/{f}'
    if not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(src):
        print('copying', f, f'({os.path.getsize(src) / 1e9:.1f} GB)'); shutil.copy(src, dst)
ARGS = f'--cache_dir {CACHE} --splits_dir {ROOT}/splits --runs_dir {ROOT}/runs --results_dir {ROOT}/results'
print('ready')

## A. Main benchmark: one part per model
4 models × 3 seeds = 12 parts. The HAVIC-only parts take ~1 min, those with the
frequency CNN take a few minutes each. `MODES_THIS_SESSION` limits how many models run now.

In [ ]:
MODES = ['havic', 'havic_emo', 'havic_freq', 'full']
MODES_THIS_SESSION = 4
for m in MODES[:MODES_THIS_SESSION]:
    print(f'========== MODEL: {m} ==========')
    !python -u evaluate_benchmark.py --protocol main {ARGS} --modes {m} | grep --line-buffered -E "^---|reuse|ep |Error|Traceback|line "

## Merge: main results table (from all saved parts)

In [ ]:
!python -u evaluate_benchmark.py --protocol main {ARGS} --merge_only | sed -n '/^# TruthShield/,$p'

## B. Leave-one-engine-out: one part per held-out generator
Train on 5 generators, test on the unseen 6th. 6 generators × 2 models (HAVIC head vs full TruthShield).
`ENGINES_THIS_SESSION` limits how many run now.

In [ ]:
ENGINES = ['kling2.5', 'pixverseV5', 'seedance1.0', 'sora2', 'veo3.1', 'wan2.5']
ENGINES_THIS_SESSION = 6
LOEO_SEEDS = '0'          # '0 1 2' for mean ± std (3x the time)
for e in ENGINES[:ENGINES_THIS_SESSION]:
    print(f'========== HELD OUT: {e} ==========')
    !python -u evaluate_benchmark.py --protocol loeo {ARGS} --modes havic full --seeds {LOEO_SEEDS} --engines {e} | grep --line-buffered -E "^---|reuse|===|Error|Traceback|line "

## Merge: leave-one-engine-out table

In [ ]:
!python -u evaluate_benchmark.py --protocol loeo {ARGS} --modes havic full --seeds {LOEO_SEEDS} --merge_only | sed -n '/^# Leave/,$p'

## C. Explainability figures (one part per generator)

In [ ]:
FIG = f'{ROOT}/results/figures'
for e in ENGINES:
    if os.path.exists(f'{FIG}/explain_{e}.png'):
        print('done:', e); continue
    !python -u explain.py --cache_dir {CACHE} --splits_dir {ROOT}/splits --ckpt {ROOT}/runs/main/full_s0.pt --out_dir {FIG} --engines {e} | grep --line-buffered -E "saved|Error|Traceback"
from IPython.display import Image, display
for f in sorted(glob.glob(f'{FIG}/*.png')):
    print(os.path.basename(f)); display(Image(f, width=1000))